In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

sns.set_palette("colorblind")
sns.set_context("talk")
sns.set_style("ticks")

%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'geopandas'

## Load data

In the 'data' folder, you can find three files:
- [OTU_to_taxon](../data/OTU_to_taxon.csv) which contains the mapping of OTUs to their respective taxonomic classifications.
- [OTU_table](../data/OTU_table.csv) which is the OTU table with sample IDs and OTU counts in the different Tara Oceans samples.
- [environmental_data](../data/environmental_data.tsv) which contains some environmental data for the Tara Oceans samples.

### OTU table

In [7]:
OTUtable = pd.read_csv("../data/OTU_table.csv", index_col=0)

# the samples belongs to two groups: surface (SRF) and deep-clorophyll maximum (DCM)
# we will use the metadata to split the samples into two groups
SRF_samples = OTUtable.columns[OTUtable.columns.str.endswith("SRF")]
DCM_samples = OTUtable.columns[OTUtable.columns.str.endswith("DCM")]

OTUtable_SRF = OTUtable[SRF_samples]
OTUtable_DCM = OTUtable[DCM_samples]

# remove _SRF and _DCM from the sample names
OTUtable_SRF.columns = OTUtable_SRF.columns.str.replace("_SRF", "")
OTUtable_DCM.columns = OTUtable_DCM.columns.str.replace("_DCM", "")

# sort the samples by their names
OTUtable_SRF = OTUtable_SRF.reindex(sorted(OTUtable_SRF.columns), axis=1)
OTUtable_DCM = OTUtable_DCM.reindex(sorted(OTUtable_DCM.columns), axis=1)

In [8]:
OTUtable_SRF

,007,008,010,011,012,014,017,018,019,020,...,191,193,194,196,201,205,206,208,209,210
0007584343baed6a66bc624ab07afa51,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
000d7f48a59463756281b4ea64af743f,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
001dedda63f65dd120ccef5ed9eff10a,0,0,0,0,0,0,0,0,0,0,...,0,0,2,0,0,0,5,0,0,1
002590d1d8d94af8dfcf5980c719d116,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
00374edaf2b110b52ffe3daea3626d01,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ffbb560ce62f5a7dd92040a2be6fbcb4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ffbec099d69da313c5253f88e48d23c9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ffe268d3373da1dd4a156f814ccc09e8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ffe8a780861ace578bc742dae5b7efa3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


### OTU to taxon mapping

In [ ]:
OTUtaxon = pd.read_csv("../data/OTU_to_taxon.csv", index_col=0, names=["OTU", "taxonomy"], header=0)

# Eukaryota|Harosa|Stramenopiles|Ochrophyta|Baci... convert to multiple columns
OTUtaxon_split = OTUtaxon["taxonomy"].str.split("|", expand=True)
OTUtaxon_split.columns = [f"level_{i+1}" for i in range(OTUtaxon_split.shape[1])]

# value of the max_level column is the last non-null value in each row
OTUtaxon_split['max_level'] = OTUtaxon_split.apply(lambda x: x.last_valid_index(), axis=1)

# keep the max_level value from the max_level column
OTUtaxon_split['last_taxon'] = OTUtaxon_split.apply(lambda row: row[row['max_level']] if pd.notnull(row['max_level']) else np.nan, axis=1)

OTUtaxon_split

,level_1,level_2,level_3,level_4,level_5,level_6,level_7,level_8,level_9,max_level,last_taxon
OTU,,,,,,,,,,,
0007218b42008b73e5c1481e4b59ec9d,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Bacillariophytina,Mediophyceae,Odontella,Odontella+sinensis,level_9,Odontella+sinensis
0007584343baed6a66bc624ab07afa51,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Bacillariophytina,Mediophyceae,Thalassiosira,Thalassiosira+tumida,level_9,Thalassiosira+tumida
000d7f48a59463756281b4ea64af743f,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,None,None,None,None,level_5,Bacillariophyta
00149657c3efed31acf6c090dc9f7d3e,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Bacillariophytina,Bacillariophyceae,Bacillariophyceae_X,Bacillariophyceae_X+sp.,level_9,Bacillariophyceae_X+sp.
0019a281ff6a88bfc9d468c828caef9c,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Coscinodiscophytina,Rhizosolenids,Leptocylindrus,Leptocylindrus+danicus var. apora B651,level_9,Leptocylindrus+danicus var. apora B651
...,...,...,...,...,...,...,...,...,...,...,...
ffe6af60722e9567b8c85932dde83aa9,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Bacillariophytina,Mediophyceae,Odontella,Odontella+sinensis,level_9,Odontella+sinensis
ffe8a780861ace578bc742dae5b7efa3,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Bacillariophytina,Mediophyceae,Chaetoceros,None,level_8,Chaetoceros
ffe993abfebd143c645d9b1517169794,Eukaryota,Harosa,Stramenopiles,Ochrophyta,Bacillariophyta,Coscinodiscophytina,Coscinodiscophytina incertae sedis,Proboscia,None,level_8,Proboscia


### Environmental data

In [22]:
environmental_data = pd.read_csv("../data/environmental_data.tsv", sep="\t", index_col=0)
environmental_data

,latitude,longitude,depth_category,depth_nominal,month,SSD,temp_woa,sal_woa,no3_woa,po4_woa,sioh4_woa,dco,dcu,dfe,dzn
station,,,,,,,,,,,,,,,
7,37.0434,1.9493,SUR,9,9,727.0,25.320400,37.243389,0.654349,0.171634,0.129361,55.321,1.129,2.158,3.596
8,38.0050,3.9899,SUR,9,9,717.0,25.628609,37.451500,0.212858,0.154929,0.584276,51.391,1.152,1.700,3.868
10,40.6541,2.8407,SUR,9,9,711.0,24.797300,37.925713,0.072777,0.103806,0.451147,60.567,1.089,3.298,3.197
11,41.6645,2.7983,SUR,9,10,NaN,21.630198,37.960928,0.007061,0.028331,0.065810,65.818,1.336,3.837,4.457
12,43.3482,7.9010,SUR,5,10,675.0,20.361670,38.178169,0.006583,0.033686,0.089598,100.297,1.040,4.290,2.368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,72.4693,-71.8920,SUR,5,10,584.5,-0.892610,31.219290,1.793274,0.440587,3.355274,107.994,0.848,1.990,1.035
206,70.9618,-53.6030,SUR,5,10,564.5,2.437630,32.896900,1.785118,0.084252,2.914919,-999.000,-999.000,-999.000,-999.000
208,69.1136,-51.5086,SUR,5,10,508.0,2.938600,33.131413,1.630793,0.000000,3.127574,-999.000,-999.000,-999.000,-999.000
